In [1]:
import pandas as pd

In [2]:
df_informalemp = pd.read_csv("labor/informalemp.csv")

In [3]:
df_informalemp.info()

<class 'pandas.DataFrame'>
RangeIndex: 54554 entries, 0 to 54553
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ref_area.label        54554 non-null  str    
 1   source.label          54554 non-null  str    
 2   indicator.label       54554 non-null  str    
 3   sex.label             54554 non-null  str    
 4   classif1.label        54554 non-null  str    
 5   time                  54554 non-null  int64  
 6   obs_value             50753 non-null  float64
 7   obs_status.label      12797 non-null  str    
 8   note_classif.label    1364 non-null   str    
 9   note_indicator.label  5386 non-null   str    
 10  note_source.label     48551 non-null  str    
dtypes: float64(1), int64(1), str(9)
memory usage: 19.6 MB


In [4]:
df_informalemp.head()

,ref_area.label,source.label,indicator.label,sex.label,classif1.label,time,obs_value,obs_status.label,note_classif.label,note_indicator.label,note_source.label
0,Afghanistan,LFS - Labour Force Survey,SDG indicator 8.3.1 - Proportion of informal e...,Total,Economic activity (Broad sector): Total,2021,86.125,NaN,NaN,NaN,Repository: ILO-STATISTICS - Micro data proces...
1,Afghanistan,LFS - Labour Force Survey,SDG indicator 8.3.1 - Proportion of informal e...,Total,Economic activity (Broad sector): Agriculture,2021,99.150,NaN,NaN,NaN,Repository: ILO-STATISTICS - Micro data proces...
2,Afghanistan,LFS - Labour Force Survey,SDG indicator 8.3.1 - Proportion of informal e...,Total,Economic activity (Broad sector): Non-agriculture,2021,73.713,NaN,NaN,NaN,Repository: ILO-STATISTICS - Micro data proces...
3,Afghanistan,LFS - Labour Force Survey,SDG indicator 8.3.1 - Proportion of informal e...,Total,Economic activity (Broad sector): Industry,2021,86.758,NaN,NaN,NaN,Repository: ILO-STATISTICS - Micro data proces...
4,Afghanistan,LFS - Labour Force Survey,SDG indicator 8.3.1 - Proportion of informal e...,Total,Economic activity (Broad sector): Services,2021,62.916,NaN,NaN,NaN,Repository: ILO-STATISTICS - Micro data proces...


We cleaned and standardized the ILO informal employment dataset to make it consistent with our panel data structure for the Capstone project. First, we filtered the data to keep only total values (both sex = Total and economic activity = Total) to ensure comparability across countries. Then, we selected only the necessary variables and renamed them to a unified format: country, year, and score. We converted year and score to numeric types, removed missing values, and reset the index. The final dataset contains 232 unique countries and a continuous yearly range from 2000 to 2026, making it ready for integration with other macroeconomic indicators in a longitudinal panel analysis.

In [5]:
df_informal = df_informalemp.copy()

df_informal = df_informal[
    (df_informal["sex.label"] == "Total") &
    (df_informal["classif1.label"] == "Economic activity (Broad sector): Total")
].copy()

In [6]:
df_informal = df_informal.rename(columns={
    "ref_area.label": "country",
    "time": "year",
    "obs_value": "informal_employment"
})

df_informal = df_informal[["country", "year", "informal_employment"]]

df_informal["year"] = df_informal["year"].astype(int)
df_informal["informal_employment"] = pd.to_numeric(
    df_informal["informal_employment"], errors="coerce"
)

In [7]:
df_informal = df_informal.rename(columns={
    "ref_area.label": "country",
    "time": "year",
    "obs_value": "informal_employment"
})

df_informal = df_informal[["country", "year", "informal_employment"]]

df_informal["year"] = df_informal["year"].astype(int)
df_informal["informal_employment"] = pd.to_numeric(
    df_informal["informal_employment"], errors="coerce"
)

In [8]:
df_informal = df_informal.rename(columns={
    "ref_area.label": "country",
    "time": "year",
    "obs_value": "informal_employment"
})

df_informal = df_informal[["country", "year", "informal_employment"]]

df_informal["year"] = df_informal["year"].astype(int)
df_informal["informal_employment"] = pd.to_numeric(
    df_informal["informal_employment"], errors="coerce"
)

In [9]:
df_informal = (
    df_informal
    .groupby(["country", "year"], as_index=False)["informal_employment"]
    .mean()
)

In [10]:
# Asegura orden
df_informal = df_informal.sort_values(["country", "year"]).reset_index(drop=True)

# Interpolar dentro de cada país
df_informal["informal_employment"] = (
    df_informal.groupby("country")["informal_employment"]
    .transform(lambda s: s.interpolate(method="linear"))
)

# Rellenar extremos (si un país empieza/termina con NaN)
df_informal["informal_employment"] = (
    df_informal.groupby("country")["informal_employment"]
    .transform(lambda s: s.ffill().bfill())
)

In [11]:
print(df_informal.shape)
print(df_informal.head())
print("Países finales:", df_informal["country"].nunique())

(3294, 3)
  country  year  informal_employment
0    APEC  2004               52.430
1    APEC  2005               51.505
2    APEC  2006               51.164
3    APEC  2007               50.723
4    APEC  2008               50.436
Países finales: 232


In [12]:
df_informal.to_parquet("cleaned_data/df_informal.parquet", index=False)